# UK Road Safety Analytics
## Notebook 01: Data Loading & Initial Inspection

**Author:** David  
**Date:** July 2026  
**Stage:** Data Understanding  

### Objective
Load and inspect the DfT STATS19 road safety data covering 2020-2024.
Three datasets will be used:
1. Collisions — every reported road accident in Great Britain
2. Casualties — every person injured or killed in those accidents
3. Vehicles — every vehicle involved in those accidents

### Data Source
Department for Transport — Road Safety Open Data  
Licence: Open Government Licence v3.0  
Coverage: 2020-2024 (5 years of validated final data)

In [2]:
import pandas as pd
import numpy as np
import os

# ── LOAD ALL THREE DATASETS ───────────────────────────────────────────────

raw_path = '../data/raw/'

# Load collisions
collisions = pd.read_csv(raw_path + 'dft-road-casualty-statistics-collision-last-5-years.csv',
                         low_memory=False)

# Load casualties
casualties = pd.read_csv(raw_path + 'dft-road-casualty-statistics-casualty-last-5-years.csv',
                         low_memory=False)

# Load vehicles
vehicles = pd.read_csv(raw_path + 'dft-road-casualty-statistics-vehicle-last-5-years.csv',
                       low_memory=False)

print("=" * 55)
print("DATASETS LOADED")
print("=" * 55)
print(f"Collisions:  {collisions.shape[0]:,} rows, {collisions.shape[1]} columns")
print(f"Casualties:  {casualties.shape[0]:,} rows, {casualties.shape[1]} columns")
print(f"Vehicles:    {vehicles.shape[0]:,} rows, {vehicles.shape[1]} columns")
print("=" * 55)

print("\nCollision columns:")
print(list(collisions.columns))

print("\nCasualty columns:")
print(list(casualties.columns))

DATASETS LOADED
Collisions:  503,475 rows, 44 columns
Casualties:  640,522 rows, 23 columns
Vehicles:    920,692 rows, 32 columns

Collision columns:
['collision_index', 'collision_year', 'collision_ref_no', 'location_easting_osgr', 'location_northing_osgr', 'longitude', 'latitude', 'police_force', 'collision_severity', 'number_of_vehicles', 'number_of_casualties', 'date', 'day_of_week', 'time', 'local_authority_district', 'local_authority_ons_district', 'local_authority_highway', 'local_authority_highway_current', 'first_road_class', 'first_road_number', 'road_type', 'speed_limit', 'junction_detail_historic', 'junction_detail', 'junction_control', 'second_road_class', 'second_road_number', 'pedestrian_crossing_human_control_historic', 'pedestrian_crossing_physical_facilities_historic', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards_historic', 'carriageway_hazards', 'urban_or_rural_area', 'd

# UK Road Safety Analytics — Project Roadmap

## The Goal
Build an end-to-end data analytics and machine learning project using 
5 years of DfT STATS19 data to analyse road accident patterns across 
Great Britain and predict accident severity.

---

## Notebook 01 — Data Loading and Inspection
- Load all 3 STATS19 datasets (collisions, casualties, vehicles)
- Inspect shapes, columns, and data types
- Decode coded values using the data guide (e.g. severity 1=Fatal, 2=Serious, 3=Slight)
- Check for missing values and data quality issues
- Merge collisions and casualties into one master dataset
- Save clean merged dataset to data/processed/
- Commit and push to GitHub

---

## Notebook 02 — Exploratory Data Analysis (5-6 charts)
- Chart 1: Accident severity breakdown (Fatal vs Serious vs Slight) 
  over 5 years — how has it changed?
- Chart 2: Accidents by time of day and day of week — when is it 
  most dangerous to be on the road?
- Chart 3: Accidents by road type and speed limit — where do the 
  worst accidents happen?
- Chart 4: Seasonal pattern — which months are most dangerous?
- Chart 5: Geographic hotspot map — which regions of Great Britain 
  have the most accidents? (choropleth or scatter map using 
  latitude/longitude)
- Chart 6: Top contributing factors — weather, lighting, road 
  surface conditions
- Add markdown finding cells after every chart
- Commit and push to GitHub

---

## Notebook 03 — Feature Engineering
- Create new variables from existing columns:
  - Is rush hour (7-9am, 4-7pm)
  - Is weekend
  - Is dark (based on light conditions)
  - Is wet road
  - Is high speed limit (60mph+)
  - Day of year (captures seasonality)
- Encode categorical variables for modelling
- Define target variable: accident severity (Fatal/Serious vs Slight)
  This becomes a binary classification problem
- Split into train (2020-2022) and test (2023-2024) sets
- Commit and push to GitHub

---

## Notebook 04 — Machine Learning Model
- Test multiple models:
  - Logistic Regression (baseline)
  - Random Forest Classifier
- Check for class imbalance (Fatal accidents are rare — this matters)
- Evaluate using:
  - Accuracy
  - Precision, Recall, F1 score
  - Confusion matrix
- Check for overfitting (train vs test gap)
- Feature importance — which factors most strongly predict severity?
- Commit and push to GitHub

---

## Notebook 05 — Conclusions and Recommendations
- Summary of all findings
- Business and policy recommendations for:
  - Department for Transport
  - Local councils
  - Insurance companies
  - Road safety campaigners
- Limitations and further research
- Data sources and Harvard references
- Commit and push to GitHub

---

## After the Notebooks
- Write GitHub README (same structure as Projects 1 and 2)
- Build project Word document with all charts embedded and 
  Harvard references
- Write LinkedIn post in personal story format
- Rewrite all finding markdowns in own words before publishing

---

## Key Differences From Projects 1 and 2
- Classification model (predicting a category) not regression 
  (predicting a number) — new skill
- Geographic/location data using latitude and longitude — new skill
- Three datasets merged together — more complex data engineering
- Class imbalance handling — new challenge
- 5 notebooks instead of 4 — slightly more complex scope

In [4]:
# ── DATA QUALITY INSPECTION — ALL THREE DATASETS ─────────────────────────

def inspect_dataset(df, name):
    print(f"\n{'=' * 55}")
    print(f"{name.upper()} — DATA QUALITY REPORT")
    print(f"{'=' * 55}")
    print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
    
    # Missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'missing_count': missing,
        'missing_pct': missing_pct.round(2)
    })
    missing_df = missing_df[missing_df['missing_count'] > 0].sort_values(
        'missing_pct', ascending=False)
    
    if len(missing_df) > 0:
        print(f"\nColumns with missing values ({len(missing_df)} columns):")
        print(missing_df.to_string())
    else:
        print("\nNo missing values found.")
    
    # Duplicates
    dupes = df.duplicated().sum()
    print(f"\nDuplicate rows: {dupes:,}")
    
    # Data types
    print(f"\nData types:")
    print(df.dtypes.value_counts())

inspect_dataset(collisions, "Collisions")
inspect_dataset(casualties, "Casualties")
inspect_dataset(vehicles, "Vehicles")


COLLISIONS — DATA QUALITY REPORT
Shape: 503,475 rows, 45 columns

Columns with missing values (5 columns):
                                 missing_count  missing_pct
local_authority_highway_current             80         0.02
location_easting_osgr                       65         0.01
location_northing_osgr                      65         0.01
longitude                                   65         0.01
latitude                                    65         0.01

Duplicate rows: 0

Data types:
int64      30
object      9
float64     6
Name: count, dtype: int64

CASUALTIES — DATA QUALITY REPORT
Shape: 640,522 rows, 23 columns

No missing values found.

Duplicate rows: 0

Data types:
int64      18
object      3
float64     2
Name: count, dtype: int64

VEHICLES — DATA QUALITY REPORT
Shape: 920,692 rows, 32 columns

No missing values found.

Duplicate rows: 0

Data types:
int64     28
object     4
Name: count, dtype: int64


In [5]:
# ── CLEAN COLLISIONS — REMOVE 65 ROWS WITH MISSING COORDINATES ────────────

print(f"Collisions before cleaning: {len(collisions):,}")

# Drop rows missing latitude/longitude — we need these for the map chart
collisions_clean = collisions.dropna(subset=['latitude', 'longitude']).copy()

# Fill missing local_authority_highway_current with unknown
collisions_clean['local_authority_highway_current'] = (
    collisions_clean['local_authority_highway_current'].fillna('Unknown')
)

print(f"Collisions after cleaning:  {len(collisions_clean):,}")
print(f"Rows removed:               {len(collisions) - len(collisions_clean):,}")

# Casualties and Vehicles are already clean — just rename for consistency
casualties_clean = casualties.copy()
vehicles_clean   = vehicles.copy()

print(f"\nCasualties: {len(casualties_clean):,} rows — no cleaning needed")
print(f"Vehicles:   {len(vehicles_clean):,} rows — no cleaning needed")

Collisions before cleaning: 503,475
Collisions after cleaning:  503,410
Rows removed:               65

Casualties: 640,522 rows — no cleaning needed
Vehicles:   920,692 rows — no cleaning needed


In [6]:
# ── DECODE KEY COLUMNS ────────────────────────────────────────────────────

# Collision severity
severity_map = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
collisions_clean['severity_label'] = collisions_clean[
    'collision_severity'].map(severity_map)

# Day of week
day_map = {1:'Sunday', 2:'Monday', 3:'Tuesday', 4:'Wednesday',
           5:'Thursday', 6:'Friday', 7:'Saturday'}
collisions_clean['day_name'] = collisions_clean['day_of_week'].map(day_map)

# Light conditions
light_map = {1:'Daylight', 4:'Darkness - lights lit',
             5:'Darkness - lights unlit', 6:'Darkness - no lighting',
             7:'Darkness - lighting unknown', -1:'Unknown'}
collisions_clean['light_label'] = collisions_clean[
    'light_conditions'].map(light_map)
collisions_clean['light_label'] = collisions_clean['light_label'].fillna('Unknown')

# Weather conditions
weather_map = {1:'Fine no high winds', 2:'Raining no high winds',
               3:'Snowing no high winds', 4:'Fine with high winds',
               5:'Raining with high winds', 6:'Snowing with high winds',
               7:'Fog or mist', 8:'Other', 9:'Unknown', -1:'Unknown'}
collisions_clean['weather_label'] = collisions_clean[
    'weather_conditions'].map(weather_map)
collisions_clean['weather_label'] = collisions_clean['weather_label'].fillna('Unknown')

# Road surface conditions
road_map = {1:'Dry', 2:'Wet or damp', 3:'Snow', 4:'Frost or ice',
            5:'Flood over 3cm deep', 6:'Oil or diesel', 7:'Mud',
            -1:'Unknown'}
collisions_clean['road_surface_label'] = collisions_clean[
    'road_surface_conditions'].map(road_map)
collisions_clean['road_surface_label'] = collisions_clean[
    'road_surface_label'].fillna('Unknown')

# Urban or rural
urban_map = {1: 'Urban', 2: 'Rural', 3: 'Unallocated'}
collisions_clean['area_label'] = collisions_clean[
    'urban_or_rural_area'].map(urban_map)

# Road type
road_type_map = {1:'Roundabout', 2:'One way street', 3:'Dual carriageway',
                 6:'Single carriageway', 7:'Slip road', 9:'Unknown',
                 12:'One way street/slip road', -1:'Unknown'}
collisions_clean['road_type_label'] = collisions_clean[
    'road_type'].map(road_type_map)
collisions_clean['road_type_label'] = collisions_clean[
    'road_type_label'].fillna('Unknown')

# Convert date to datetime
collisions_clean['date'] = pd.to_datetime(collisions_clean['date'])
collisions_clean['month'] = collisions_clean['date'].dt.month
collisions_clean['month_name'] = collisions_clean['date'].dt.strftime('%B')
collisions_clean['hour'] = collisions_clean['time'].str[:2].astype(float)

print("Decoding complete. Sample of key decoded columns:")
print(collisions_clean[['collision_severity', 'severity_label',
                         'day_of_week', 'day_name',
                         'light_conditions', 'light_label',
                         'weather_conditions', 'weather_label']].head(8).to_string())


/var/folders/hs/ky3tw1815mdg4wxxvn3tpg0h0000gn/T/ipykernel_73035/3343217062.py:54: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  collisions_clean['date'] = pd.to_datetime(collisions_clean['date'])


Decoding complete. Sample of key decoded columns:
   collision_severity severity_label  day_of_week   day_name  light_conditions             light_label  weather_conditions       weather_label
0                   3         Slight            7   Saturday                 6  Darkness - no lighting                   1  Fine no high winds
1                   3         Slight            4  Wednesday                 1                Daylight                   1  Fine no high winds
2                   3         Slight            3    Tuesday                 4   Darkness - lights lit                   1  Fine no high winds
3                   3         Slight            5   Thursday                 4   Darkness - lights lit                   1  Fine no high winds
4                   3         Slight            2     Monday                 1                Daylight                   1  Fine no high winds
5                   3         Slight            3    Tuesday                 1              

In [7]:
# ── FIX DATE PARSING WARNING ──────────────────────────────────────────────
collisions_clean['date'] = pd.to_datetime(
    collisions_clean['date'], dayfirst=True)
collisions_clean['month'] = collisions_clean['date'].dt.month
collisions_clean['month_name'] = collisions_clean['date'].dt.strftime('%B')
collisions_clean['hour'] = collisions_clean['time'].str[:2].astype(float)

print("Date parsing fixed.")
print(f"Date range: {collisions_clean['date'].min().date()} to "
      f"{collisions_clean['date'].max().date()}")
print(f"Hour range: {collisions_clean['hour'].min()} to "
      f"{collisions_clean['hour'].max()}")

Date parsing fixed.
Date range: 2020-01-01 to 2024-12-31
Hour range: 0.0 to 23.0


In [8]:
# ── MERGE COLLISIONS AND CASUALTIES ──────────────────────────────────────

# Merge on collision_index — each casualty links back to one collision
merged = pd.merge(
    casualties_clean,
    collisions_clean,
    on='collision_index',
    how='inner',
    suffixes=('_casualty', '_collision')
)

print(f"Casualties:          {len(casualties_clean):,}")
print(f"Collisions (clean):  {len(collisions_clean):,}")
print(f"Merged rows:         {len(merged):,}")
print(f"Merged columns:      {merged.shape[1]}")

# Decode casualty severity
merged['casualty_severity_label'] = merged['casualty_severity'].map(
    {1: 'Fatal', 2: 'Serious', 3: 'Slight'})

# Decode casualty type
casualty_type_map = {
    1: 'Pedestrian', 2: 'Cyclist', 3: 'Motorcycle 50cc and under',
    4: 'Motorcycle 125cc and under', 5: 'Motorcycle over 125cc and up to 500cc',
    6: 'Motorcycle over 500cc', 7: 'Taxi/Private hire car',
    8: 'Car occupant', 9: 'Minibus occupant', 10: 'Bus or coach occupant',
    11: 'Horse rider', 12: 'Agricultural vehicle occupant',
    13: 'Tram occupant', 14: 'Van/Goods vehicle occupant',
    15: 'Goods vehicle occupant', 16: 'Mobility scooter rider',
    17: 'Electric motorcycle rider', 18: 'Other vehicle occupant',
    19: 'Unknown vehicle occupant', 20: 'Motorcycle - unknown cc',
    21: 'Wheelchair user', 90: 'Other', 97: 'Motorcycle - unknown',
    98: 'Unknown', 99: 'Unknown'
}
merged['casualty_type_label'] = merged['casualty_type'].map(
    casualty_type_map).fillna('Other')

# Decode sex of casualty
merged['sex_label'] = merged['sex_of_casualty'].map(
    {1: 'Male', 2: 'Female', -1: 'Unknown', 9: 'Unknown'})

print(f"\nCasualty severity breakdown:")
print(merged['casualty_severity_label'].value_counts())

print(f"\nTop 5 casualty types:")
print(merged['casualty_type_label'].value_counts().head(5))

Casualties:          640,522
Collisions (clean):  503,410
Merged rows:         640,449
Merged columns:      76

Casualty severity breakdown:
casualty_severity_label
Slight     511158
Serious    121337
Fatal        7954
Name: count, dtype: int64

Top 5 casualty types:
casualty_type_label
Minibus occupant                         344129
Other                                     98816
Pedestrian                                77979
Motorcycle 50cc and under                 41511
Motorcycle over 125cc and up to 500cc     21332
Name: count, dtype: int64


In [9]:
# ── CHECK RAW CASUALTY TYPE VALUES ───────────────────────────────────────

print("Raw casualty_type value counts:")
print(merged['casualty_type'].value_counts().head(15))

print("\nRaw sex_of_casualty value counts:")
print(merged['sex_of_casualty'].value_counts())

print("\nRaw casualty_class value counts:")
print(merged['casualty_class'].value_counts())

Raw casualty_type value counts:
casualty_type
 9     344129
 0      89162
 1      77979
 3      41511
 5      21332
 19     18437
 11     10035
 4       8944
 8       7270
-1       4934
 2       4666
 90      2307
 21      2306
 97      1673
 22      1223
Name: count, dtype: int64

Raw sex_of_casualty value counts:
sex_of_casualty
 1    392566
 2    242061
-1      5740
 9        82
Name: count, dtype: int64

Raw casualty_class value counts:
casualty_class
1    428177
2    123110
3     89162
Name: count, dtype: int64


In [10]:
# ── READ THE ACTUAL DATA GUIDE TO GET CORRECT CODES ──────────────────────

import openpyxl

guide_path = '../data/raw/dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2024.xlsx'

# Read all sheet names first
xl = pd.ExcelFile(guide_path)
print("Sheets in data guide:")
for sheet in xl.sheet_names:
    print(f"  {sheet}")

Sheets in data guide:
  2024_code_list


In [11]:
# ── READ THE DATA GUIDE AND FIND CASUALTY TYPE CODES ─────────────────────

guide = pd.read_excel(guide_path, sheet_name='2024_code_list')

print(f"Shape: {guide.shape}")
print(f"\nColumns: {list(guide.columns)}")
print(f"\nFirst 10 rows:")
print(guide.head(10).to_string())

Shape: (1820, 5)

Columns: ['table', 'field name', 'code/format', 'label', 'note']

First 10 rows:
       table              field name code/format                label                                                                                                                                                                       note
0  collision         collision_index         NaN                  NaN  unique value for each collision. The collision_index combines the collision_year and collision_ref_no to form a unique ID. It can be used to join to Vehicle and Casualty
1  collision          collision_year         NaN                  NaN                                                                                                                                                                        NaN
2  collision        collision_ref_no         NaN                  NaN                               In year id used by the police to reference a collision. It is not unique outsi

In [12]:
# ── FIND CASUALTY TYPE CODES FROM OFFICIAL GUIDE ─────────────────────────

casualty_type_guide = guide[guide['field name'] == 'casualty_type']
print("Official casualty_type codes from DfT data guide:")
print(casualty_type_guide[['code/format', 'label']].to_string(index=False))

Official casualty_type codes from DfT data guide:
code/format                                                     label
          0                                                Pedestrian
          1                                                   Cyclist
          2              Motorcycle 50cc and under rider or passenger
          3             Motorcycle 125cc and under rider or passenger
          4 Motorcycle over 125cc and up to 500cc rider or  passenger
          5                  Motorcycle over 500cc rider or passenger
          8                            Taxi/Private hire car occupant
          9                                              Car occupant
         10                 Minibus (8 - 16 passenger seats) occupant
         11             Bus or coach occupant (17 or more pass seats)
         16                                               Horse rider
         17                             Agricultural vehicle occupant
         18                             

In [13]:
 # ── FIX CASUALTY TYPE DECODE USING OFFICIAL GUIDE ────────────────────────

casualty_type_map_correct = {
    0:   'Pedestrian',
    1:   'Cyclist',
    2:   'Motorcycle 50cc and under',
    3:   'Motorcycle 125cc and under',
    4:   'Motorcycle 125cc to 500cc',
    5:   'Motorcycle over 500cc',
    8:   'Taxi/Private hire car',
    9:   'Car occupant',
    10:  'Minibus occupant',
    11:  'Bus or coach occupant',
    16:  'Horse rider',
    17:  'Agricultural vehicle occupant',
    18:  'Tram occupant',
    19:  'Van/Goods vehicle under 3.5t',
    20:  'Goods vehicle 3.5t to 7.5t',
    21:  'Goods vehicle over 7.5t',
    22:  'Mobility scooter rider',
    23:  'Electric motorcycle rider',
    90:  'Other vehicle occupant',
    97:  'Motorcycle unknown cc',
    98:  'Goods vehicle unknown weight',
    99:  'Unknown vehicle type',
    -1:  'Unknown'
}

merged['casualty_type_label'] = merged['casualty_type'].map(
    casualty_type_map_correct).fillna('Other/Historic')

print("CORRECTED casualty type breakdown:")
print(merged['casualty_type_label'].value_counts().head(10))

print("\nQuick sense check:")
print(f"Car occupants:  {(merged['casualty_type_label'] == 'Car occupant').sum():,}")
print(f"Pedestrians:    {(merged['casualty_type_label'] == 'Pedestrian').sum():,}")
print(f"Cyclists:       {(merged['casualty_type_label'] == 'Cyclist').sum():,}")

CORRECTED casualty type breakdown:
casualty_type_label
Car occupant                    344129
Pedestrian                       89162
Cyclist                          77979
Motorcycle 125cc and under       41511
Motorcycle over 500cc            21332
Van/Goods vehicle under 3.5t     18437
Bus or coach occupant            10035
Motorcycle 125cc to 500cc         8944
Taxi/Private hire car             7270
Unknown                           4934
Name: count, dtype: int64

Quick sense check:
Car occupants:  344,129
Pedestrians:    89,162
Cyclists:       77,979


In [14]:
# ── VERIFY OTHER KEY DECODE MAPS AGAINST OFFICIAL GUIDE ──────────────────

fields_to_check = ['collision_severity', 'light_conditions',
                   'weather_conditions', 'road_surface_conditions',
                   'urban_or_rural_area', 'road_type', 'casualty_class']

for field in fields_to_check:
    field_guide = guide[guide['field name'] == field]
    if len(field_guide) > 0:
        print(f"\n{field}:")
        print(field_guide[['code/format', 'label']].to_string(index=False))


collision_severity:
code/format   label
          1   Fatal
          2 Serious
          3  Slight

light_conditions:
code/format                        label
          1                     Daylight
          4        Darkness - lights lit
          5      Darkness - lights unlit
          6       Darkness - no lighting
          7  Darkness - lighting unknown
         -1 Data missing or out of range

weather_conditions:
code/format                        label
          1           Fine no high winds
          2        Raining no high winds
          3        Snowing no high winds
          4            Fine + high winds
          5         Raining + high winds
          6         Snowing + high winds
          7                  Fog or mist
          8                        Other
          9                      Unknown
         -1 Data missing or out of range

road_surface_conditions:
code/format                        label
          1                          Dry
          2  

In [15]:
# ── VERIFY OTHER KEY DECODE MAPS AGAINST OFFICIAL GUIDE ──────────────────

fields_to_check = ['collision_severity', 'light_conditions',
                   'weather_conditions', 'road_surface_conditions',
                   'urban_or_rural_area', 'road_type', 'casualty_class',
                   'sex_of_casualty', 'day_of_week', 'speed_limit']

for field in fields_to_check:
    field_guide = guide[guide['field name'] == field]
    if len(field_guide) > 0:
        print(f"\n{'=' * 45}")
        print(f"FIELD: {field}")
        print(field_guide[['code/format', 'label']].to_string(index=False))
    else:
        print(f"\n{'=' * 45}")
        print(f"FIELD: {field} — not found in guide (free text or numeric)")


FIELD: collision_severity
code/format   label
          1   Fatal
          2 Serious
          3  Slight

FIELD: light_conditions
code/format                        label
          1                     Daylight
          4        Darkness - lights lit
          5      Darkness - lights unlit
          6       Darkness - no lighting
          7  Darkness - lighting unknown
         -1 Data missing or out of range

FIELD: weather_conditions
code/format                        label
          1           Fine no high winds
          2        Raining no high winds
          3        Snowing no high winds
          4            Fine + high winds
          5         Raining + high winds
          6         Snowing + high winds
          7                  Fog or mist
          8                        Other
          9                      Unknown
         -1 Data missing or out of range

FIELD: road_surface_conditions
code/format                        label
          1                   

In [16]:
# ── CHECK REMAINING FIELDS ────────────────────────────────────────────────

for field in ['sex_of_casualty', 'day_of_week']:
    field_guide = guide[guide['field name'] == field]
    if len(field_guide) > 0:
        print(f"\n{field}:")
        print(field_guide[['code/format', 'label']].to_string(index=False))
    else:
        print(f"\n{field} — not found in guide")


sex_of_casualty:
code/format                        label
          1                         Male
          2                       Female
          9      unknown (self reported)
         -1 Data missing or out of range

day_of_week:
code/format     label
          1    Sunday
          2    Monday
          3   Tuesday
          4 Wednesday
          5  Thursday
          6    Friday
          7  Saturday


### Data Guide Validation — All Decode Maps Confirmed

All coded fields were cross-checked against the official DfT Open Dataset
Data Guide (2024 edition) before use in analysis. One error was caught and
corrected during this process:

- **casualty_type code 9** was initially mapped to "Minibus occupant"
  (based on an older STATS19 specification). The 2024 official guide
  confirms code 9 = "Car occupant". This correction affected 344,129 rows
  and would have significantly distorted any casualty type analysis if
  left uncorrected.

This cross-validation step is standard professional practice when working
with coded government datasets. All other decode maps (collision severity,
light conditions, weather, road surface, urban/rural, road type, casualty
class, sex, day of week) matched the official guide exactly.

In [17]:
# ── SAVE CLEAN MERGED DATASET ─────────────────────────────────────────────

import os
os.makedirs('../data/processed/', exist_ok=True)

# Select the most useful columns for analysis
cols_to_keep = [
    # Identifiers
    'collision_index', 'collision_year', 'date', 'month',
    'month_name', 'day_of_week', 'day_name', 'hour',

    # Location
    'latitude', 'longitude', 'urban_or_rural_area', 'area_label',
    'local_authority_district', 'police_force',

    # Collision characteristics
    'collision_severity', 'severity_label',
    'number_of_vehicles', 'number_of_casualties',
    'speed_limit', 'road_type', 'road_type_label',
    'light_conditions', 'light_label',
    'weather_conditions', 'weather_label',
    'road_surface_conditions', 'road_surface_label',

    # Casualty characteristics
    'casualty_severity', 'casualty_severity_label',
    'casualty_class', 'casualty_type', 'casualty_type_label',
    'sex_of_casualty', 'sex_label',
    'age_of_casualty', 'age_band_of_casualty',
]

# Keep only columns that exist in merged
cols_to_keep = [c for c in cols_to_keep if c in merged.columns]

clean = merged[cols_to_keep].copy()

clean.to_csv('../data/processed/road_safety_clean.csv', index=False)

print("=" * 55)
print("NOTEBOOK 01 COMPLETE — CLEAN DATASET SAVED")
print("=" * 55)
print(f"Rows:             {len(clean):,}")
print(f"Columns:          {clean.shape[1]}")
print(f"Years covered:    2020 to 2024")
print(f"Severity split:")
print(f"  Fatal:          {(clean['severity_label']=='Fatal').sum():,}")
print(f"  Serious:        {(clean['severity_label']=='Serious').sum():,}")
print(f"  Slight:         {(clean['severity_label']=='Slight').sum():,}")
print(f"Saved to:         data/processed/road_safety_clean.csv")
print("=" * 55)
print("NEXT: Notebook 02 — Exploratory Data Analysis")
print("=" * 55)

NOTEBOOK 01 COMPLETE — CLEAN DATASET SAVED
Rows:             640,449
Columns:          35
Years covered:    2020 to 2024
Severity split:
  Fatal:          12,389
  Serious:        149,011
  Slight:         479,049
Saved to:         data/processed/road_safety_clean.csv
NEXT: Notebook 02 — Exploratory Data Analysis
